# **Quest 7: Not Fast but Furious**

![Header image](./imgs/image_quest_7.svg)
[<img src="./imgs/instagram.svg" alt="My SVG" width="15" height="15"><small>_Monika Lipińska_</small>](https://www.instagram.com/monli_art/)

## **Part I**

Counting, searching, analysing data… it’s time for action!  
Every knight must be in peak physical condition — and so must their steed. Together, they form a single team, and the upcoming tournament event will put that partnership to the test.  
It is time for the **chariot races**!

These races are run in groups, but this is no ordinary competition.  
It’s **not** about who reaches the finish line first — all chariots move at **exactly the same speed**.  
Instead, the challenge is to collect **magical essence** from the Debugging Spirits Forest.

Each chariot is equipped with a special device that gathers essence from each segment of the track. The device has a **power level**, and the amount of essence collected from a segment equals the device’s current power.

Each device follows a **plan** consisting of actions:

- `+` increase power by 1
- `-` decrease power by 1
- `=` maintain current power

When the plan reaches its final action, it simply **loops back to the beginning**.

All devices:

- start with **power = 10**
- execute one action **before entering each segment**
- can **never** drop below power 0
  - if power is 0 and the action is `-`, nothing happens, but the action still counts

The test round for the squires is about to begin — a perfect chance to ensure you understand the rules.  
Each participant may choose **any** plan from the provided list (_your notes_).  
The track length is **10 segments**.

Your task:

### **Determine the ranking of the plans, from the one that collects the most essence to the one that collects the least.**

---

## **Example**

### **Plans**

```
A:+,-,=,=
B:+,=,-,+
C:=,-,+,+
D:=,=,=,+
```

### **Step 1 — Repeat each plan to cover all 10 segments**

| Segment | 1   | 2   | 3   | 4   | 5   | 6   | 7   | 8   | 9   | 10  |
| ------- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| **A**   | +   | -   | =   | =   | +   | -   | =   | =   | +   | -   |
| **B**   | +   | =   | -   | +   | +   | =   | -   | +   | +   | =   |
| **C**   | =   | -   | +   | +   | =   | -   | +   | +   | =   | -   |
| **D**   | =   | =   | =   | +   | =   | =   | =   | +   | =   | =   |

### **Step 2 — Compute power per segment**

| Segment | 1   | 2   | 3   | 4   | 5   | 6   | 7   | 8   | 9   | 10  |
| ------- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| **A**   | 11  | 10  | 10  | 10  | 11  | 10  | 10  | 10  | 11  | 10  |
| **B**   | 11  | 11  | 10  | 11  | 12  | 12  | 11  | 12  | 13  | 13  |
| **C**   | 10  | 9   | 10  | 11  | 11  | 10  | 11  | 12  | 12  | 11  |
| **D**   | 10  | 10  | 10  | 11  | 11  | 11  | 11  | 12  | 12  | 12  |

### **Step 3 — Total essence collected**

- **A:** 103
- **B:** 116
- **C:** 107
- **D:** 110

### **Final ranking**

$$
\textbf{BDCA}
$$

---

## **Your Task**

**What is the ranking of the squires’ action plans after 10 segments?**

---


In [1]:
from collections import deque
from functools import reduce
from itertools import cycle, islice

from more_itertools import first, last, one
import networkx as nx
import matplotlib.pyplot as plt
from util import Str
from test_utilities import test

tests = [
    {
        "name": "Example",
        "s": """
            A:+,-,=,=
            B:+,=,-,+
            C:=,-,+,+
            D:=,=,=,+
        """,
        "expected": "BDCA",
    },
]


@test(tests=tests[:])
def part_I(s: str, segments: int = 10) -> str:
    devices = [l.strip().split(":")[0] for l in s.strip().splitlines()]
    action_plans = [l.strip().split(":")[1].split(",") for l in s.strip().splitlines()]

    essence = [10] * len(devices)
    actions = {"+": lambda e: e + 1, "-": lambda e: e - 1, "=": lambda e: e}

    for i, ess in enumerate(essence):
        som = 0
        for a in islice(cycle(action_plans[i]), segments):
            ess = actions[a](ess)
            som += ess
        essence[i] = som

    indices = sorted(range(len(devices)), key=essence.__getitem__, reverse=True)

    return "".join(devices[i] for i in indices)


Test Example passed, for part_I.
Success


In [2]:
with open("../inputs/everybody_codes_e2024_q07_p1.txt") as f:
    notes1 = f.read()

print(f"Part I: {part_I(notes1)}")

Part I: FGCEAIJKD


## **Part II**

It’s time for the **knights’ races**!

Unlike the squires, the knights compete on **special tracks** that interfere with the devices’ behaviour, sometimes overriding their planned actions. The track forms a **closed loop**, allowing knights to complete multiple laps. Every knight begins at segment **`S`** and moves clockwise (to the right when viewed from above).

---

## **Track Symbols**

Each segment of the track affects the device as follows:

- **`=`** — execute the plan’s action normally
- **`+`** — force **increase power by 1**, ignoring the plan
- **`-`** — force **decrease power by 1**, ignoring the plan
- **`S`** — start/finish segment; behaves like `=` (does not override the plan)

---

## **Execution Rules**

- Devices start with **power = 10**
- Before entering each segment, the device performs **one action**
- The **first action** is executed at the **first segment after `S`**
- The **last action** of each loop is executed **on the `S` segment**
- If the plan ends, it **repeats from the beginning**
- Power **cannot drop below 0**
  - If power is 0 and the action is `-`, nothing happens, but the action still counts

The race lasts **10 loops**.

---

## **Example**

Given plans:

```
A:+,-,=,=
B:+,=,-,+
C:=,-,+,+
D:=,=,=,+
```

And the sample track:

```
S+===
-   +
=+=-+
```

The overridden actions for the first loop become:

| Seg | Track | A   | B   | C   | D   |
| --- | ----- | --- | --- | --- | --- |
| 1   | +     | +   | +   | +   | +   |
| 2   | =     | -   | =   | -   | =   |
| 3   | =     | =   | -   | +   | =   |
| 4   | =     | =   | +   | +   | +   |
| 5   | +     | +   | +   | +   | +   |
| 6   | +     | +   | =   | +   | =   |
| 7   | -     | -   | -   | -   | -   |
| 8   | =     | =   | +   | +   | +   |
| 9   | +     | +   | +   | +   | +   |
| 10  | =     | -   | =   | -   | =   |
| 11  | -     | -   | -   | -   | -   |
| 12  | S     | =   | +   | +   | +   |

After computing power per segment and summing:

- **A:** 129
- **B:** 148
- **C:** 154
- **D:** 158

### **Ranking after 1 loop:**

**DCBA**

### **Ranking after 10 loops:**

**DCBA**

With totals:

- **A:** 1290
- **B:** 3640
- **C:** 3700
- **D:** 4280

---

## **Your Track for Part II**

The racetrack for the knights’ race is:

```
S-=++=-==++=++=-=+=-=+=+=--=-=++=-==++=-+=-=+=-=+=+=++=-+==++=++=-=-=--
-                                                                     -
=                                                                     =
+                                                                     +
=                                                                     +
+                                                                     =
=                                                                     =
-                                                                     -
--==++++==+=+++-=+=-=+=-+-=+-=+-=+=-=+=--=+++=++=+++==++==--=+=++==+++-
```

---

## **Your Task**

**What is the final ranking of the action plans after 10 loops?**

---


In [3]:
from collections import deque
from functools import reduce
from itertools import cycle, islice

from more_itertools import count_cycle, first, last, one
import networkx as nx
import matplotlib.pyplot as plt
from util import Str
from test_utilities import test

tests = [
    {
        "name": "Example after 1 segment",
        "notes": """
            A:+,-,=,=
            B:+,=,-,+
            C:=,-,+,+
            D:=,=,=,+
        """,
        "racetrack": """
            S+===
            -   +
            =+=-+
        """,
        "loops": 1,
        "expected": "DCBA",
    },
    {
        "name": "Example",
        "notes": """
            A:+,-,=,=
            B:+,=,-,+
            C:=,-,+,+
            D:=,=,=,+
        """,
        "racetrack": """
            S+===
            -   +
            =+=-+
        """,
        "loops": 10,
        "expected": "DCBA",
    },
]


def get_next(
    row: int, col: int, rt: list[str], seen: set[tuple[int, int]]
) -> tuple[int, int]:
    rows, cols = len(rt), len(rt[0])
    return first(
        (row + dr, col + dc)
        for dr, dc in ((-1, 0), (0, 1), (1, 0), (0, -1))
        if 0 <= row + dr < rows
        and 0 <= col + dc < cols
        and rt[row + dr][col + dc] != " "
        and (row + dr, col + dc) not in seen
    )


def parse_racetrack(racetrack: str) -> list[str]:
    rt = [l.lstrip() for l in racetrack.strip().splitlines()]
    temp = []
    seen = set()
    row, col = get_next(0, 0, rt, seen)

    while rt[row][col] != "S":
        seen.add((row, col))
        temp.append((rt[row][col]))
        row, col = get_next(row, col, rt, seen)

    temp.append(rt[row][col])
    return temp


@test(tests=tests[:])
def part_II(notes: str, racetrack: str, loops: int = 10) -> str:
    devices = [l.strip().split(":")[0] for l in notes.strip().splitlines()]
    action_plans = [
        l.strip().split(":")[1].split(",") for l in notes.strip().splitlines()
    ]

    essence = [10] * len(devices)
    actions = {"+": lambda e: e + 1, "-": lambda e: e - 1, "=": lambda e: e}

    rt = parse_racetrack(racetrack)

    for i, ess in enumerate(essence):
        som = 0
        race_itr = zip(map(last, count_cycle(rt, n=loops)), cycle(action_plans[i]))

        for c, (t, a) in enumerate(race_itr, start=1):
            if t == "+":
                ess += 1
            elif t == "-":
                ess -= 1
            else:
                ess = actions[a](ess)

            som += ess
        essence[i] = som

    indices = sorted(range(len(devices)), key=essence.__getitem__, reverse=True)

    return "".join(devices[i] for i in indices)


Test Example after 1 segment passed, for part_II.
Test Example passed, for part_II.
Success


In [4]:
with open("../inputs/everybody_codes_e2024_q07_p2.txt") as f:
    notes2 = f.read()

racetrack = """
S-=++=-==++=++=-=+=-=+=+=--=-=++=-==++=-+=-=+=-=+=+=++=-+==++=++=-=-=--
-                                                                     -
=                                                                     =
+                                                                     +
=                                                                     +
+                                                                     =
=                                                                     =
-                                                                     -
--==++++==+=+++-=+=-=+=-+-=+-=+-=+=-=+=--=+++=++=+++==++==--=+=++==+++-
"""

print(f"PartII: {part_II(notes2, racetrack)}")

PartII: IGBAHJFEC


#### Part III

You have advanced to the **final duel**!

The general rules of the chariot devices remain the same, but this time the **racetrack is far more complex**. Each knight must design an **action plan** for their device, with the following strict requirements:

- The plan must be **exactly 11 actions long**
- It must contain:
  - **5** actions of `+`
  - **3** actions of `-`
  - **3** actions of `=`

For example, a valid action plan is:

```text
+-=+-=+-=++
```

Your rival knight declares their strategy first, and you quickly note it down.  
The duel is set for **2024 loops** of the track.

Victory feels almost certain—but just to be sure, you decide to compare a few plans.  
And then, to be _truly_ prepared, you decide to consider **all possible valid plans** that meet the constraints above.

---

#### Final Duel Racetrack

The racetrack for the final duel is:

```text
S+= +=-== +=++=     =+=+=--=    =-= ++=     +=-  =+=++=-+==+ =++=-=-=--
- + +   + =   =     =      =   == = - -     - =  =         =-=        -
= + + +-- =-= ==-==-= --++ +  == == = +     - =  =    ==++=    =++=-=++
+ + + =     +         =  + + == == ++ =     = =  ==   =   = =++=
= = + + +== +==     =++ == =+=  =  +  +==-=++ =   =++ --= + =
+ ==- = + =   = =+= =   =       ++--          +     =   = = =--= ==++==
=     ==- ==+-- = = = ++= +=--      ==+ ==--= +--+=-= ==- ==   =+=    =
-               = = = =   +  +  ==+ = = +   =        ++    =          -
-               = + + =   +  -  = + = = +   =        +     =          -
--==++++==+=+++-= =-= =-+-=  =+-= =-= =--   +=++=+++==     -=+=++==+++-
```

---

### Your Task

**How many different winning action plans can you prepare?**

(A winning plan is one that collects more total essence than your rival’s plan over 2024 loops of this track.)


In [5]:
from collections.abc import Iterator
from itertools import combinations


def plans(nr_plus=5, nr_minus=3, nr_equal=3) -> Iterator[list[str]]:
    n = nr_plus + nr_minus + nr_equal

    for plusses in combinations(range(n), r=nr_plus):
        rest = set(range(n)) - set(plusses)
        for mins in combinations(rest, nr_minus):
            yield [
                "+" if i in plusses else "-" if i in mins else "=" for i in range(11)
            ]


def eval(loops, actions, rt, plan):
    som = 0
    ess = 10
    race_itr = zip(map(last, count_cycle(rt, n=loops)), cycle(plan))

    for c, (t, a) in enumerate(race_itr, start=1):
        if t == "+":
            ess += 1
        elif t == "-":
            ess -= 1
        else:
            ess = actions[a](ess)

        som += ess
    return som


def smart_eval(loops, actions, rt, plan):
    n = len(plan)
    total = eval(n, actions, rt, plan)
    total2 = eval(2 * n, actions, rt, plan)

    a = abs(total - 2 * total2) // 2
    b = total - a
    k = loops // len(plan)

    return a * k * k + b * k


def part_III(notes: str, racetrack: str, loops: int = 10) -> int:
    devices = [l.strip().split(":")[0] for l in notes.strip().splitlines()]
    action_plans = [
        l.strip().split(":")[1].split(",") for l in notes.strip().splitlines()
    ]

    actions = {"+": lambda e: e + 1, "-": lambda e: e - 1, "=": lambda e: e}

    rt = parse_racetrack(racetrack)
    reference = smart_eval(loops, actions, rt, one(action_plans))

    count_better = 0

    for plan in plans():
        som = smart_eval(loops, actions, rt, plan)
        if som > reference:
            count_better += 1

    return count_better

In [ ]:
with open("../inputs/everybody_codes_e2024_q07_p3.txt") as f:
    notes3 = f.read()

racetrack = """
S+= +=-== +=++=     =+=+=--=    =-= ++=     +=-  =+=++=-+==+ =++=-=-=--
- + +   + =   =     =      =   == = - -     - =  =         =-=        -
= + + +-- =-= ==-==-= --++ +  == == = +     - =  =    ==++=    =++=-=++
+ + + =     +         =  + + == == ++ =     = =  ==   =   = =++=       
= = + + +== +==     =++ == =+=  =  +  +==-=++ =   =++ --= + =           
+ ==- = + =   = =+= =   =       ++--          +     =   = = =--= ==++==
=     ==- ==+-- = = = ++= +=--      ==+ ==--= +--+=-= ==- ==   =+=    =
-               = = = =   +  +  ==+ = = +   =        ++    =          -
-               = + + =   +  -  = + = = +   =        +     =          -
--==++++==+=+++-= =-= =-+-=  =+-= =-= =--   +=++=+++==     -=+=++==+++-"""

print(f"Part III {part_III(notes3, racetrack, loops=2024)}")  # slow 3m27.1s!!

Part III 3924


![happy](./imgs/happy_quack.svg)
